# 02 — Label integrity and leakage audit

**Purpose:** examine the target definition, class distribution, and suspiciously target-aligned fields.

This notebook treats `anomaly` as the candidate target because that is the target column observed in the supplied CSV. It does **not** assume that the labels are independently or correctly generated.

## 1. Load data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset.csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (78600, 14)


,timestamp,hour_of_day,sending_address,receiving_address,amount,transaction_type,location_region,ip_prefix,login_frequency,session_duration,purchase_pattern,age_group,risk_score,anomaly
0,2022-04-11 12:47:27,12,0x9d32d0bf2c00f41ce7ca01b66e174cc4dcb0c1da,0x39f82e1c09bc6d7baccc1e79e5621ff812f50572,796.949206,transfer,Europe,192.000,3,48,focused,established,18.75,low_risk
1,2022-06-14 19:12:46,19,0xd6e251c23cbf52dbd472f079147873e655d8096f,0x51e8fbe24f124e0e30a614e14401b9bbfed5384c,0.010000,purchase,South America,172.000,5,61,focused,established,25.00,low_risk
2,2022-01-18 16:26:59,16,0x2e0925b922fed01f6a85d213ae2718f54b8ca305,0x52c7911879f783d590af45bda0c0ef2b8536706f,778.197390,purchase,Asia,192.168,3,74,focused,established,31.25,low_risk
3,2022-06-15 09:20:04,9,0x93efefc25fcaf31d7695f28018d7a11ece55457f,0x8ac3b7bd531b3a833032f07d4e47c7af6ea7bace,300.838358,transfer,South America,172.000,8,111,high_value,veteran,36.75,low_risk
4,2022-02-18 14:35:30,14,0xad3b8de45d63f5cce28aef9a82cf30c397c6ceb9,0x6fdc047c2391615b3facd79b4588c7e9106e49f2,775.569344,sale,Africa,172.160,6,100,high_value,veteran,62.50,moderate_risk


## 2. Target distribution and basic validity

In [2]:
TARGET = "anomaly"
if TARGET not in df.columns:
    raise KeyError(f"Expected target column {TARGET!r}; found {df.columns.tolist()}")

print("Target missing:", int(df[TARGET].isna().sum()))
display(df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count"))
display((df[TARGET].value_counts(normalize=True, dropna=False) * 100).round(3).rename("percent").to_frame())

Target missing: 0


,count
anomaly,
low_risk,63494
moderate_risk,8611
high_risk,6495


,percent
anomaly,
low_risk,80.781
moderate_risk,10.955
high_risk,8.263


## 3. Numeric feature versus target summaries

In [3]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET]
for col in numeric_cols:
    print(f"\\n### {col}")
    display(df.groupby(TARGET, dropna=False)[col].agg(["count","min","median","mean","max","std"]))

\n### hour_of_day


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,0,12.0,11.641109,23,6.992830
low_risk,63494,0,12.0,12.134564,23,6.752026
moderate_risk,8611,0,5.0,7.012426,23,6.554707


\n### amount


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,0.01,491.667467,495.352890,1393.027468,243.012623
low_risk,63494,0.01,486.027151,486.214159,1557.150905,238.435717
moderate_risk,8611,0.01,701.855115,628.659679,1485.989339,265.112407


\n### ip_prefix


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,10.0,172.16,149.367603,192.168,68.200001
low_risk,63494,10.0,172.16,147.483633,192.168,69.496558
moderate_risk,8611,10.0,172.16,147.530346,192.168,69.466905


... (2 outputs left out)

\n### session_duration


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,20,30.0,29.462510,39,5.747063
low_risk,63494,20,59.0,67.574133,159,38.166378
moderate_risk,8611,20,118.0,115.584601,159,29.400959


\n### risk_score


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,90.0,100.0000,97.686374,100.0000,3.422143
low_risk,63494,15.0,35.4375,36.274932,59.0625,11.834191
moderate_risk,8611,62.5,67.5000,69.200427,84.3750,6.774497


## 4. Categorical feature versus target cross-tabs

In [4]:
categorical_cols = [c for c in df.columns if c != TARGET and
                    (df[c].dtype == "object" or df[c].nunique(dropna=False) <= 30)]
for col in categorical_cols:
    print(f"\\n### {col}")
    ct = pd.crosstab(df[col].fillna("<MISSING>").astype(str),
                     df[TARGET].fillna("<MISSING>").astype(str),
                     normalize="index")
    display(ct.round(3))

\n### timestamp


anomaly,high_risk,low_risk,moderate_risk
timestamp,,,
2022-01-01 00:00:45,0.0,1.0,0.0
2022-01-01 00:12:24,0.0,1.0,0.0
2022-01-01 00:14:54,0.0,1.0,0.0
2022-01-01 00:27:12,0.0,1.0,0.0
2022-01-01 00:36:27,0.0,0.0,1.0
...,...,...,...
2022-12-31 23:25:21,0.0,1.0,0.0
2022-12-31 23:36:26,0.0,1.0,0.0
2022-12-31 23:41:28,0.0,1.0,0.0


\n### hour_of_day


anomaly,high_risk,low_risk,moderate_risk
hour_of_day,,,
0,0.092,0.663,0.245
1,0.083,0.654,0.263
10,0.081,0.867,0.052
11,0.085,0.860,0.055
12,0.083,0.869,0.048
13,0.084,0.868,0.049
14,0.084,0.868,0.048
15,0.082,0.863,0.055
16,0.081,0.863,0.056


\n### sending_address


anomaly,high_risk,low_risk,moderate_risk
sending_address,,,
0x00262fa3b21847775e2db42965d0192e8dfe5bdc,0.092,0.816,0.092
0x002bcd751410a69b3bce463890abb2fcab81a755,0.069,0.759,0.172
0x00318c7963a45ceb6b3101e1794ad42137da8f3f,0.069,0.828,0.103
0x003ebce509d14e6252dc14c35beb43d2dc52f35c,0.034,0.897,0.069
0x004fd71e61d0fb00f974635faec45195b8239607,0.078,0.784,0.138
...,...,...,...
0xff209e4a1ce114ec3e245cf48fa19eeaff9cec31,0.069,0.793,0.138
0xff4ad17b86a1a52511777aab39db4c94fa4f1753,0.138,0.776,0.086
0xffa670245089044b1e355508a7843692a25a5e52,0.138,0.750,0.112


... (10 outputs left out)

\n### purchase_pattern


anomaly,high_risk,low_risk,moderate_risk
purchase_pattern,,,
focused,0.000,1.000,0.000
high_value,0.000,0.689,0.311
random,0.248,0.737,0.015


\n### age_group


anomaly,high_risk,low_risk,moderate_risk
age_group,,,
established,0.000,1.000,0.000
new,0.248,0.737,0.015
veteran,0.000,0.689,0.311


## 5. Investigate the previously flagged fields

In [5]:
for col in ["risk_score", "transaction_type", "purchase_pattern", "age_group"]:
    if col not in df.columns:
        continue
    print(f"\\n### {col}")
    if pd.api.types.is_numeric_dtype(df[col]):
        display(df.groupby(TARGET)[col].agg(["min","max","mean","median","nunique"]))
        ranges = df.groupby(TARGET)[col].agg(["min","max"])
        print("Class range overlap must be assessed from these intervals.")
    else:
        display(pd.crosstab(df[col], df[TARGET], margins=True))

\n### risk_score


,min,max,mean,median,nunique
anomaly,,,,,
high_risk,90.0,100.0000,97.686374,100.0000,3
low_risk,15.0,59.0625,36.274932,35.4375,24
moderate_risk,62.5,84.3750,69.200427,67.5000,4


Class range overlap must be assessed from these intervals.
\n### transaction_type


anomaly,high_risk,low_risk,moderate_risk,All
transaction_type,,,,
phishing,2546,0,0,2546
purchase,0,24940,0,24940
sale,0,16429,8611,25040
scam,3949,0,0,3949
transfer,0,22125,0,22125
All,6495,63494,8611,78600


\n### purchase_pattern


anomaly,high_risk,low_risk,moderate_risk,All
purchase_pattern,,,,
focused,0,26033,0,26033
high_value,0,18205,8217,26422
random,6495,19256,394,26145
All,6495,63494,8611,78600


\n### age_group


anomaly,high_risk,low_risk,moderate_risk,All
age_group,,,,
established,0,26033,0,26033
new,6495,19256,394,26145
veteran,0,18205,8217,26422
All,6495,63494,8611,78600


## 6. Single-feature predictability screen (diagnostic only)

In [6]:
# This is a diagnostic screen, not proof of leakage. High predictability can be
# legitimate, target-derived, or caused by synthetic label construction.
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str)
if y.nunique() == 2:
    y_enc = LabelEncoder().fit_transform(y)
    scoring = "roc_auc"
else:
    y_enc = LabelEncoder().fit_transform(y)
    scoring = "roc_auc_ovr"

rows = []
for col in X.columns:
    xcol = X[[col]].copy()
    if pd.api.types.is_numeric_dtype(xcol[col]):
        prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), [col])])
    else:
        prep = ColumnTransformer([("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore")), [col])])
    model = make_pipeline(prep, RandomForestClassifier(
        n_estimators=150, min_samples_leaf=5, class_weight="balanced",
        random_state=42, n_jobs=-1))
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    try:
        scores = cross_val_score(model, xcol, y_enc, cv=cv, scoring=scoring, n_jobs=-1)
        rows.append({"feature": col, "mean_cv_score": scores.mean(), "std_cv_score": scores.std()})
    except Exception as e:
        rows.append({"feature": col, "mean_cv_score": np.nan, "std_cv_score": np.nan, "error": str(e)})

screen = pd.DataFrame(rows).sort_values("mean_cv_score", ascending=False)
display(screen)

,feature,mean_cv_score,std_cv_score
12,risk_score,1.000000,0.000000
5,transaction_type,0.936294,0.000803
9,session_duration,0.807864,0.001896
8,login_frequency,0.803401,0.001147
10,purchase_pattern,0.802405,0.001097
11,age_group,0.802405,0.001097
1,hour_of_day,0.614567,0.003298
4,amount,0.557509,0.002591
7,ip_prefix,0.500461,0.002161
6,location_region,0.500302,0.002229


## 7. Leakage decision register

Fill this table after tracing how each field was created and when it became available.

| Field | Available at prediction time? | Derived from target/risk rule? | Decision | Evidence |
|---|---|---|---|---|
| risk_score | Unknown | Suspected | Hold out pending provenance | Perfect class separation observed in initial inspection |
| transaction_type | Unknown | Suspected proxy | Investigate | Strong class-specific pattern |
| purchase_pattern | Unknown | Unknown | Investigate | Check construction logic |
| age_group | Unknown | Unknown | Investigate | Check whether synthetic rules assigned labels by age group |

**Do not interpret a high cross-validation score as proof of genuine predictive power.** A feature may leak the target or encode the label-generation rule.

## Interpretation checklist

1. Obtain the label-generation code or written rules.
2. Determine the exact moment each feature becomes available.
3. Establish whether `risk_score` is an input, an output, or a transformation of the target rule.
4. Separate direct leakage, proxy leakage, and legitimate predictive association.
5. Do not finalize the feature list until provenance is documented.